# Environmental Features and Crash Frequency in Philadelphia

A street-segment study of Philadelphia crash frequency, road design, tree canopy, grade, and related public data. It documents the research question, source data, modeling findings, exploratory data-processing decisions, and the handoff from notebook analysis to the browser map in `app/`.

The notebook does not reproduce every app implementation detail. The [final web app](https://benpolinsky.github.io/PHLCRSH/) uses DuckDB-WASM to read exported GeoParquet files and run the map/story queries locally in the browser.


## Executive Summary

This project analyzes automobile crash data in Philadelphia, PA to determine if there's a correlation between crashes and road grade (elevation) or tree canopy. Both vector data (e.g. PennDOT crashes, road geometry) and raster data (elevation and tree canopy coverage) were used. To prepare the data, a significant amount of effort was dedicated to determining the width and grade of city road segments. Post-processing, statistical analysis and regression were run to identify any correlation between crashes and grading or tree cover across road segments.

For road widths, curbs and cartway data were joined to centerline data. Transects were drawn perpendicular to centerlines and measured until hitting curb geometry, handling edge cases due to intersections or divided roads with islands. Road grading was calculated by sampling points along a centerline, obtaining corresponding values in elevation raster data and then taking the median. The resulting widths seem reasonable for much of Philadelphia's roads, though outliers with impossible grades were identified. Most suspicious segments were _near_ large elevation changes, a data processing error that needs to be fixed.

Using negative binomial models, three regressions were run: non-natural factors vs crashes per centerline; then adding natural factors vs crashes per centerline; then natural factors with interactions vs crashes per centerline. Each successive model proved more robust, with the third demonstrating tree canopy coverage's apparent protective relationship was strongest on narrower streets and weakened as cartway width increased. Grade was associated with fewer crashes at lower-speed contexts, but the grade-speed interaction was positive, suggesting steepness becomes less protective, and potentially hazardous, as speeds rise.

It should be noted that data here has not been vigorously tested for accuracy. This project's assertions are preliminary.



## Note on Process and Rejected Routes

This notebook keeps the successful data-processing path in executable code and summarizes rejected routes in short notes. Commented-out code is only retained where it documents an attempted route that was not carried into the final workflow.


## Environment and Data Expectations

The notebook expects the `roads` Python environment and the local project data folders:

- `raw-data/` for downloaded source datasets.
- `transformed-data/` for projected and intermediate geospatial files.
- `philly_segments.parquet` and `philly_block_groups.parquet` for the app-ready exports.

Core Python packages used here include `pandas`, `geopandas`, `numpy`, `rasterio`, `matplotlib`, `seaborn`, `pygris`, `census`, and `requests`. Census API access should be provided with `CENSUS_API_KEY`; do not hardcode a key in the notebook.


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from rasterio.plot import show
import matplotlib.pyplot as plt
import seaborn as sns
import shapely
from shapely import make_valid
from shapely.geometry import LineString, Point
from shapely.ops import unary_union

import pygris
import census

import requests
import os

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2


# transforms df to 2272 and writes it to specified filepath
# if filepath exists, read it and return
def to_2272_file(df, filepath):
    df_2272 = None;
    
    if not os.path.exists(filepath):
        print("write file")
        df_2272 = df.to_crs(2272)
        df_2272.to_file(filepath, driver="geoJSON")
    else:
        print("read file")
        df_2272 = gpd.read_file(filepath)

    return df_2272
    
crash_data = gpd.read_file("raw-data/PennDOT Crash Data/collision_crash_2020_2024.geojson")

## Data Acquisition and Setup

### 2024 PennDOT Crash Data for Philadelphia

The crash data comes from the Philadelphia subset of PennDOT annual crash records published through OpenDataPhilly. This section checks severity fields and basic completeness before joining crashes to street segments.

#### Data Quality

##### What does `max_severity_level` mean?

0 - No injury

1 - Fatal injury
2 - Suspected serious injury - obvious because the person cannot move or operate normally
3 - Suspected minor injury - evident to others, but not as serious as above: bruises, bleeding, etc.
4 - Possible injury - not evident but reported; could include being knocked out or internal injuries

8 - Injury, unknown severity
9 - Unknown

We're going to need to finesse this a bit. We could shuffle "No Injury" to 5 and have a severity scale. That kind of works, though "Possible injury" gives me a bit of pause because it could seemingly be as severe as 3, perhaps 2.

We could try to scale, but I'm leaning more toward using these as categorical rather than ordinal.


In [ ]:
print(crash_data.columns)
print(crash_data.crs) # 4326

print(crash_data["max_severity_level"].describe())
print(crash_data["max_severity_level"].value_counts(dropna=False))

sns.countplot(x="max_severity_level", data=crash_data)
plt.show()

Other fields of interest:

Weather1 and Weather2 have lots of NAs. I do not want to assume blanks mean fine conditions without stronger documentation.

Illumination is not very helpful, but `illumination_dark` is interesting: 0 = no and 1 = yes.

Time of day and hour of day are not reliable enough. We'll have to drop them. Police arrival time might have more signal, but still not enough for this version.


In [ ]:

# most do not have weather data. Are we to interpret that as fine conditions.
print(crash_data.describe())
print(crash_data["weather1"].value_counts(dropna=False))
print(crash_data["weather2"].value_counts(dropna=False))

crashes_no_weather = crash_data[["weather1", "weather2"]].notna().any(axis=1).sum()
print(f"crashes without weather1 or weather2 {crashes_no_weather}")

print(crash_data["illumination"].describe())
print(crash_data["illumination"].value_counts(dropna=False))
print(crash_data["illumination_dark"].value_counts(dropna=False))

# neither time_of_day nor hour_of_day are usable - too many NaNs
print(crash_data["hour_of_day"].value_counts(dropna=False))
print(crash_data["time_of_day"].value_counts(dropna=False))
print(crash_data["arrival_tm"].value_counts(dropna=False)) # maybe

In [ ]:
# while we can whittle our data down a bit, and drop what we do not need, we want to be sure to convert to 2272
print(crash_data.crs)
print(crash_data.centroid)

transformed_crash_path = "transformed-data/PennDOT Crash Data/crash_data_2272.geojson"
crash_data_2272 = to_2272_file(crash_data, transformed_crash_path)

print(crash_data_2272.crs)
print(crash_data_2272.centroid)

crash_data_2272.head()

### Centerline Street Data

Lane count: **we'll have to look elsewhere for lane count.**

Directionality: `oneway` values are B for both directions, FT for from-to digitization direction, and TF for the opposite.

Classification: `class` needs some cleaning. Here are the classifications:

```
0-Navy Yard
1-Expressways
2-Major Arterial
3-Minor Arterial
4-Collector
5-Local
6-Driveway
9-Low Speed Ramps
10-High Speed Ramps
12-Non Travable
14-City Boundary
13-???
15-Walking Connector
18-Traffic Controlled Crosswalks
```

We might not want to consider street types that are not driveable or have a near-zero possibility of crashes, like Driveways, Non-Travable, City Boundary, Walking Connectors, and Traffic Controlled Crosswalks. Since I can't find a great source for every class, I'm going to plot and classify the data so I can see for myself, then make a call one way or another.

12 looks like things such as Forbidden Drive and walkways, though maintenance vehicles may access some of them.

Class 13 is a mixed bag. Legacy? At any rate, if we use this classification as a predictor, then we should probably exclude it.

Compare later to the Vision Zero High Injury Network centerline source: https://opendataphilly.org/datasets/street-centerlines-for-vision-zero-high-injury-network-2017/


In [ ]:

# centerline

center_line_data = gpd.read_file("raw-data/Street Centerline Data/Street_Centerline.geojson")

print(center_line_data.crs) # 4326

center_line_data.columns

print("oneway")
print(center_line_data["oneway"].value_counts())

print("pre_dir")
print(center_line_data["pre_dir"].value_counts())

print("seg_id")
print(center_line_data["seg_id"])

print("st_type")
print(center_line_data["st_type"])

print("class")

print(center_line_data.columns)

center_line_data["class"].value_counts()

In [ ]:

thirteens = center_line_data[center_line_data["class"] == 13]
thirteens.explore()

# these are all sorts of driveways, shopping centers, parks, universities. Again, I'm split on whether or not to include them.
driveways = center_line_data[center_line_data["class"] == 6]
driveways.explore()

## before we manipulate, let's transform so we don't have to do it multiple times
cl_2272_filepath = "transformed-data/Street Centerline Data/2272_centerlines.geojson"
center_line_2272 = to_2272_file(center_line_data, cl_2272_filepath)

exclude_classes = [6, 12, 13, 14, 15, 18]
# could show maps of others
center_line_subset = center_line_2272[~center_line_2272["class"].isin(exclude_classes)]
center_line_excluded = center_line_2272[center_line_2272["class"].isin(exclude_classes)]

center_line_master_2722 = center_line_subset.copy()
print(center_line_master_2722)
center_line_subset.explore(column="class", categorical=True, legend=True, cmap="tab10")


Indeed - we understand the centerline data better, but at the risk of not having speed limit or num of lanes. We'll definitely want speed limits.


### Curbs, Cartways, and Road Width

Curbs/cartways are the source geometry for street-width estimation. The useful FCODE groups are:

- 1000 and 1001: cartway or paved travelway polygons.
- 1010-1012 and 1030-1032: concrete or grass islands, useful for understanding divided roads.

The final width calculation does not try to assign an entire cartway polygon to one street. Instead, it starts from each centerline segment and measures cross-sections through the cartway polygons.

Multipart cartways and divided roads are handled at the transect level. If a cross-section intersects multiple paved cartway pieces because an island, median, or multipart polygon splits the road, the code keeps those pieces separately as `n_parts`. It records `paved_width_ft` as the sum of the paved pieces, while `span_width_ft` measures the outermost-to-outermost span across all pieces. For the final model/app width, I use `cartway_paved_median_ft`, which avoids counting the physical island/median gap as paved road width.


In [ ]:
# curbs with cartways

curbs_with_cartways = gpd.read_file("raw-data/Curbs and Cartways/Curbs.geojson")
curbs_with_cartways_2272 = to_2272_file(curbs_with_cartways, "transformed-data/Curbs and Cartways/Curbs.geojson")

print(curbs_with_cartways_2272.head(10))
print(curbs_with_cartways_2272.crs) # 4326

print(curbs_with_cartways_2272.columns.tolist())
print(curbs_with_cartways_2272.head())
print(curbs_with_cartways_2272['fcode'].value_counts())

roadway_ids = [1000, 1001]
island_ids = [1010,1011,1012,1030,1031,1032]
    
roadways = curbs_with_cartways_2272[curbs_with_cartways_2272["fcode"].isin(roadway_ids)]
islands = curbs_with_cartways_2272[curbs_with_cartways_2272["fcode"].isin(island_ids)]

islands.plot(column="fcode", categorical=True, legend=True, figsize=(10,10))
plt.show() # this looks correct - Roosevelt expressway; 76 + 95; airport + cargo city area.

In [ ]:

## State roads - let's see if we can incorporate

rmsseg = gpd.read_file("raw-data/PennDOT State Roads/RMSSEG_(State_Roads).geojson")

rmsseg = rmsseg[rmsseg["CTY_CODE"].astype(str) == "67"].to_crs(2272)  # Philly county code
rmsseg.columns.to_list()
rmsseg["is_divided"] = (
    rmsseg["DIVSR_TYPE"].notna()
    & ~rmsseg["DIVSR_TYPE"].astype(str).str.strip().isin(["", "0", "00"])
) | (rmsseg["DIVSR_WIDTH"].fillna(0) > 0)

# bonus: keep the type for downstream modeling
rmsseg["median_type"] = rmsseg["DIVSR_TYPE"]
rmsseg.groupby("DIVSR_TYPE").size().plot()
rmsseg.plot(column="DIVSR_TYPE", categorical=True, legend=True, figsize=(25,25)) # 0 = not divided, this is state roads only of course
plt.show()
rmsseg["DIVSR_TYPE"].value_counts(dropna=False)
# first go is here, but we're seeing some false positives,. so we've gotta distinguish a bit more here
# https://gis.penndot.gov/BPR_PDF_FILES/Documents/Traffic/Highway_Statistics/HPMS/Field_Manual/2016_PA_Field_Manual.pdf has the keys 

rmsseg["is_divided"] =  rmsseg["DIVSR_TYPE"].astype(int) > 1 # still needs refining I think... need to keep on
plt.show()
rmsseg.plot(column="is_divided", categorical=True, legend=True, cmap="flag", figsize=(25,25)) # 0 = not divided, this is state roads only of course



The PennDOT state-road attributes are useful for lanes, total width, and divided-road context, but they do not replace the cartway-width calculation. Many Philadelphia streets are not state roads, so the main width variable needs to be derived from city cartway and centerline geometry. The above state data can be used to complement or validate derived data.


### Final Cartway Width Calculation: Centerline Transects

For each driveable centerline segment, I generated perpendicular transects at regular stations along the line, intersected those transects with cartway polygons, and summarized the measured widths by segment.

Process:

1. Filter curb/cartway polygons to paved travelway FCODEs 1000 and 1001.
2. Assign each centerline a temporary `cl_id`.
3. Walk along each centerline every 75 feet, trimming 50 feet from each end where possible to reduce intersection artifacts.
4. Estimate the local tangent from two nearby points on the centerline.
5. Rotate that tangent into a perpendicular unit vector: `normal = (-dy / d, dx / d)`.
6. Build a 250-foot cross-section centered on the station point.
7. Intersect the cross-section with cartway polygons.
8. Extract every line piece from the transect/cartway intersection. Multipart cartways or divided roads can produce multiple pieces on one transect.
9. Measure `paved_width_ft` as the sum of paved pieces, `span_width_ft` as the outermost-to-outermost span, and `n_parts` as the number of pieces.
10. Keep valid transects between 4 and 250 feet, then summarize per segment with median width and a confidence flag.

This route is implemented in `data_prep/widths.py` and enters the final analytical table through `compute_segment_widths(cl)`. The model/app field is `cartway_width_ft = cartway_paved_median_ft`.


In [ ]:
ROADWAY_FCODES = [1000, 1001]
ISLAND_FCODES = [1010, 1011, 1012, 1030, 1031, 1032]

roadways = curbs_with_cartways_2272[curbs_with_cartways_2272["fcode"].isin(ROADWAY_FCODES)].copy()
islands = curbs_with_cartways_2272[curbs_with_cartways_2272["fcode"].isin(ISLAND_FCODES)].copy()

def prep_gdf(gdf, epsg=2272):
    out = gdf.to_crs(epsg).copy()
    out = out[out.geometry.notna() & ~out.geometry.is_empty].copy()
    out["geometry"] = out.geometry.map(lambda g: make_valid(g) if not g.is_valid else g)
    return out[out.geometry.notna() & ~out.geometry.is_empty].copy()

def line_parts(g):
    """Extract line pieces from an intersection geometry. Points add no width."""
    if g is None or g.is_empty:
        return []
    if g.geom_type == "LineString":
        return [g] if g.length > 0 else []
    if g.geom_type == "MultiLineString":
        return [p for p in g.geoms if p.length > 0]
    if g.geom_type == "GeometryCollection":
        parts = []
        for p in g.geoms:
            parts.extend(line_parts(p))
        return parts
    return []


In [ ]:
def make_perpendicular_transects(
    centerlines, id_col="cl_id", spacing_ft=75, trim_ft=50,
    half_len_ft=125, tangent_delta_ft=5,
):
    rows = []
    for _, row in centerlines.iterrows():
        line = row.geometry
        if line is None or line.is_empty or line.length <= 0:
            continue

        L = float(line.length)
        trim = min(float(trim_ft), max(0.0, (L - 1.0) / 2.0))

        if L - 2 * trim < max(1.0, spacing_ft):
            stations = np.array([L / 2.0])
        else:
            stations = np.arange(trim, L - trim + 1e-9, float(spacing_ft))
            if len(stations) == 0:
                stations = np.array([L / 2.0])

        for j, s in enumerate(stations):
            p = line.interpolate(float(s))
            s0 = max(0.0, float(s) - float(tangent_delta_ft))
            s1 = min(L, float(s) + float(tangent_delta_ft))
            if s1 <= s0:
                continue

            a = line.interpolate(s0)
            b = line.interpolate(s1)
            dx, dy = b.x - a.x, b.y - a.y
            d = np.hypot(dx, dy)
            if d == 0:
                continue

            # Perpendicular unit vector. This is the key calculation:
            # tangent = (dx, dy), normal = (-dy, dx), then normalize.
            nx, ny = -dy / d, dx / d
            transect = LineString([
                (p.x - half_len_ft * nx, p.y - half_len_ft * ny),
                (p.x + half_len_ft * nx, p.y + half_len_ft * ny),
            ])
            rows.append({
                id_col: row[id_col],
                "transect_no": j,
                "station_ft": float(s),
                "geometry": transect,
            })

    return gpd.GeoDataFrame(rows, geometry="geometry", crs=centerlines.crs)


In [ ]:
def measure_transect_widths(transects, surfaces, id_col="cl_id", grid_size_ft=None):
    """Measure cartway width along each transect.

    paved_width_ft = total length of transect pieces inside cartway polygons.
    span_width_ft = outermost-to-outermost cartway span along the transect.
    """
    if transects.empty:
        return gpd.GeoDataFrame(
            columns=[id_col, "paved_width_ft", "span_width_ft"],
            geometry="geometry", crs=transects.crs,
        )

    surfaces = surfaces.to_crs(transects.crs)
    pairs = surfaces.sindex.query(transects.geometry, predicate="intersects")

    groups = {}
    if pairs.size:
        pair_df = pd.DataFrame({"tx_pos": pairs[0], "surf_pos": pairs[1]})
        for tx_pos, g in pair_df.groupby("tx_pos", sort=False):
            groups[int(np.asarray(tx_pos).item())] = g["surf_pos"].to_numpy(dtype=np.int64)

    records = []
    tx_geoms = transects.geometry.to_numpy()
    surface_geoms = surfaces.geometry.reset_index(drop=True)

    for tx_pos, t in enumerate(tx_geoms):
        surf_pos = groups.get(tx_pos)
        if surf_pos is None or len(surf_pos) == 0:
            paved_width = np.nan
            span_width = np.nan
            n_parts = 0
            center_covered = False
        else:
            surf = unary_union(list(surface_geoms.iloc[surf_pos].values))
            inter = shapely.intersection(t, surf, grid_size=grid_size_ft) if grid_size_ft else t.intersection(surf)
            parts = line_parts(inter)
            paved_width = float(sum(p.length for p in parts)) if parts else np.nan
            n_parts = len(parts)

            if parts:
                projected_ends = []
                for p in parts:
                    projected_ends.append(t.project(Point(p.coords[0])))
                    projected_ends.append(t.project(Point(p.coords[-1])))
                span_width = float(max(projected_ends) - min(projected_ends))
            else:
                span_width = np.nan

            center_covered = bool(surf.covers(t.interpolate(t.length / 2.0)))

        records.append({
            id_col: transects.iloc[tx_pos][id_col],
            "transect_no": transects.iloc[tx_pos]["transect_no"],
            "station_ft": transects.iloc[tx_pos]["station_ft"],
            "paved_width_ft": paved_width,
            "span_width_ft": span_width,
            "n_parts": n_parts,
            "center_covered": center_covered,
            "geometry": t,
        })

    return gpd.GeoDataFrame(records, geometry="geometry", crs=transects.crs)


In [ ]:
def summarize_widths(raw_transects, id_col="cl_id", prefix="cartway", min_width_ft=4, max_width_ft=250):
    x = raw_transects.copy()
    x["valid_width"] = (
        x["span_width_ft"].notna()
        & x["span_width_ft"].between(min_width_ft, max_width_ft)
    )
    good = x[x["valid_width"]].copy()

    all_counts = x.groupby(id_col).size().rename(f"{prefix}_n_transects")
    valid_counts = good.groupby(id_col).size().rename(f"{prefix}_n_valid")

    stats = good.groupby(id_col).agg(
        span_median_ft=("span_width_ft", "median"),
        paved_median_ft=("paved_width_ft", "median"),
        parts_median=("n_parts", "median"),
        center_covered_share=("center_covered", "mean"),
    ).add_prefix(prefix + "_")

    summary = all_counts.to_frame().join(valid_counts).join(stats)
    summary[f"{prefix}_valid_share"] = (
        summary[f"{prefix}_n_valid"] / summary[f"{prefix}_n_transects"]
    )
    return summary.reset_index()

def widths_by_centerline(centerlines, surfaces, id_col="cl_id", spacing_ft=75, trim_ft=50, half_len_ft=125):
    cl = centerlines.copy()
    if id_col not in cl.columns:
        cl = cl.reset_index(drop=True)
        cl[id_col] = cl.index

    cl = prep_gdf(cl)
    surfaces = prep_gdf(surfaces)

    cl_parts = cl.explode(index_parts=False).reset_index(drop=True)
    cl_parts = cl_parts[cl_parts.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

    transects = make_perpendicular_transects(
        cl_parts, id_col=id_col, spacing_ft=spacing_ft, trim_ft=trim_ft, half_len_ft=half_len_ft,
    )
    raw_t = measure_transect_widths(transects, surfaces, id_col=id_col)
    summary = summarize_widths(raw_t, id_col=id_col)
    out = cl.merge(summary, on=id_col, how="left")
    return out, raw_t, transects

# Full run used by the final table builder:
# cl_for_widths = center_line_subset.reset_index(drop=True).copy()
# cl_for_widths["cl_id"] = cl_for_widths.index
# full_out, raw_width_transects, width_transects = widths_by_centerline(cl_for_widths, roadways)
# full_out["cartway_width_ft"] = full_out["cartway_paved_median_ft"]


In [ ]:
# Full run used by the final table builder. Speed:
# data_prep.widths.compute_segment_widths caches the output to a GPKG.
cl_for_widths = center_line_subset.reset_index(drop=True).copy()
cl_for_widths["cl_id"] = cl_for_widths.index

full_out, raw_width_transects, width_transects = widths_by_centerline(
    cl_for_widths,
    roadways,
    spacing_ft=75,
    trim_ft=50,
    half_len_ft=125,
)

full_out["cartway_width_ft"] = full_out["cartway_paved_median_ft"]
full_out["width_confidence"] = np.select(
    [
        full_out["cartway_n_valid"].fillna(0) >= 3,
        full_out["cartway_n_valid"].fillna(0) == 2,
        full_out["cartway_n_valid"].fillna(0) == 1,
    ],
    ["good", "okay", "low_one_or_two_transects"],
    default="missing",
)

widths_for_final_table = full_out[[
    "seg_id",
    "cartway_width_ft",
    "cartway_paved_median_ft",
    "cartway_span_median_ft",
    "cartway_valid_share",
    "cartway_n_valid",
    "width_confidence",
]].copy()

widths_for_final_table.head()


### Rejected Width Routes

I tried a few routes before the final transect approach:

- A direct roadway-polygon to centerline spatial join over-associated roads around intersections, islands, and divided roads.
- Polygon-level minimum-rotated-rectangle and area/length widths were useful sanity checks, but they measured cartway polygons rather than centerline segments.
- A buffered roadway-to-centerline match scored candidates by overlap and distance, but diagnostics were worse than the final centerline-first transect method.

The important distinction is that the final route starts from the centerline, cuts across the cartway, measures what that cross-section intersects, and then summarizes repeated measurements. That makes the width value segment-level by construction.


### Traffic Calming Devices

This is exploratory. I inspect install dates and segment identifiers, then filter to devices installed before 2025 so future planned records do not leak into the analysis window.


In [ ]:

# calming devices

traffic_calming_devices = gpd.read_file("raw-data/Traffic Calming Devices/traffic_calming_devices.geojson")
traffic_calming_2272 = to_2272_file(traffic_calming_devices, "transformed-data/Traffic Calming Devices/traffic_calming_devices_2272.geojson")

print("Calming: \n")
print(traffic_calming_devices.columns)
print(traffic_calming_devices.head(10))
print(traffic_calming_devices.crs) # 4326
# TODO: get unique values and add readable map
print(traffic_calming_devices.id.unique())
print(traffic_calming_devices.seg_id)
print(traffic_calming_devices.install_dt.sort_values()) # we have up to 2026 so need to filter down

traffic_calming_devices_through_2024 = traffic_calming_devices[traffic_calming_devices["install_dt"] < "2025-01-01"]
print(traffic_calming_devices_through_2024.install_dt.sort_values())


### Intersection Controls 

Intersection controls are divided into three types: "Conventional", "All Way", and "Signalized". The first two are stop signs. "All Way" means all ways have a stop sign, at least we hope. Conventional means a non-all way stop, and signalized mean there's some sort of traffic signal. Unfortunately, for these last two controls we have no idea _which road_ gets the stop sign or signal, though we can imagine it's the "more main" or heavier trafficked. For this analysis, we'll have to treat each type as a binary indicator - present or not. But present at what? There's no way to tell intersections from our centerline data alone. For this analysis, we'll buffer the intersection controls by 5m, and then see which centerlines intersect.


**Attempted route: direct intersection-control join.**

A direct spatial join from intersection controls to centerlines was considered, but it does not identify which approach road actually receives the stop sign or signal. The note remains to explain why this input stayed exploratory.


In [ ]:

intersection_controls = gpd.read_file("raw-data/Intersection Controls/Intersection_Controls.geojson")
intersection_controls_2272 = to_2272_file(intersection_controls, "transformed-data/Intersection Controls/Intersection_Controls_2272.geojson")
print("intersections: \n")
print(intersection_controls_2272.columns)
print(intersection_controls_2272.head(10))
print(intersection_controls_2272.crs)
print(intersection_controls_2272.describe())
print(intersection_controls_2272.stoptype.unique())

# need to join now to... centerlines is the master
# I will siphon them off until im sure i can build it all up i guess
buf = intersection_controls_2272.copy().buffer(55)
buf.explore()
# Attempted route: direct intersection-control-to-centerline join.
# This was parked because the controls identify the intersection control type,
# but not which approach road receives the stop or signal.
# center_line_master_2722_ic_test = center_line_master_2722.sjoin(intersection_controls_2272, "left", "intersects")
# center_line_master_2722_ic_test.head(10)

In [ ]:
# DVRPC Traffic counts

print("traffic counts: \n")

traffic_counts_path = "raw-data/DVRPC Traffic Count/dvrpc_2024_philly_traffic_counts.geojson"

if not os.path.exists("raw-data/DVRPC Traffic Count/dvrpc_2024_philly_traffic_counts.geojson"):
    url = "https://arcgis.dvrpc.org/portal/rest/services/transportation/trafficcounts/FeatureServer/0/query"
    params = {
        "where": "setyear=2024 AND co_name LIKE 'Phil%'",
        "outsr": 4326,
        "outfields": "*",
        "f": "geojson"
    }
    r = requests.get(url, params=params)
    counts = gpd.read_file(r.text)
    counts.to_file("raw-data/DVRPC Traffic Count/dvrpc_2024_philly_traffic_counts.geojson", driver="GeoJSON")
else: 
    counts = gpd.read_file(traffic_counts_path)

print(counts.head(10))
print(counts.columns)
print(counts.crs) # 4326

In [ ]:


# Tree canopy data - CRS = 2272
# according to the site - 1 is tree canopy
with rasterio.open("raw-data/Phila Land Cover Raster/PPR_LandCover_2018.gdb") as src:
    print(src.crs) # 2722
    print(src.bounds)
    print(src.res) # 0.5 x 0.5
    print(src.shape)
    print(src.transform)
    print(src.tags())
     
    # downsample in out_shape
    data = src.read(1, out_shape=(src.height // 25, src.width // 25))

    # mask to canopy only
    canopy = np.where(data == 1, 1, 0) # why no y?

    plt.figure(figsize=(10,10))
    plt.imshow(canopy, cmap="Greens")
    plt.title("2018 TC PH")
    plt.axis("off")
    
    plt.show()

### Elevation and Road Grade Calculation

Elevation comes from the Philadelphia 3-foot 2022 DEM. The final grade workflow is centerline-based, like the width workflow: sample elevation along each street segment, smooth the sampled profile, and summarize the grade per segment.

I tried two simpler routes first. Endpoint-only sampling was too sensitive to short segments and bad endpoint hits. Buffered zonal statistics around each segment produced noisy min/max ranges and too many unusable values. The final route samples points along the centerline itself and applies a rolling median to reduce bridge, overpass, and one-point raster artifacts.


In [ ]:
print("Elevation raster")
elevation_raster_path = "raw-data/Phila Elevation Raster/Philadelphia_dem_3ft_2022.tif"
NODATA_THRESHOLD = -1e30

with rasterio.open(elevation_raster_path) as src:
    print(src.crs)
    print(src.res)
    print(src.bounds)
    print(src.nodata)  # float32 minimum sentinel, about -3.4e38

    # Downsample only for display, not for grade calculation.
    data = src.read(1, out_shape=(src.height // 10, src.width // 10))
    data_masked = np.ma.masked_less(data, NODATA_THRESHOLD)

plt.figure(figsize=(10, 10))
im = plt.imshow(data_masked, cmap="terrain")
plt.colorbar(im, label="Elevation (ft)")
plt.axis("off")
plt.show()


### Rejected Grade Routes

- **Endpoint sampling:** sample the first and last point of each segment, then compute `abs(start_elev - end_elev) / length`. This was intuitive and sometimes plausible, but produced extreme grades when either endpoint hit a bridge, wall, overpass, or bad raster edge.
- **Buffered zonal statistics:** buffer each segment and compute DEM min/max/range or percentile range inside the buffer. This captured too much surrounding terrain, especially near steep embankments and infrastructure, and was noisier than centerline sampling.

Those attempts were useful because they exposed the main failure mode: suspicious high-grade segments clustered near bridges, overpasses, ramps, Vine Street, and sharp elevation breaks.


In [ ]:
N_SAMPLES = 10
SMOOTH_WINDOW = 5

def sample_profile_along_line(line: LineString, src, n: int = N_SAMPLES):
    rows = []
    for i in range(n):
        frac = i / (n - 1)
        station_ft = frac * line.length
        p = line.interpolate(frac, normalized=True)
        val = next(src.sample([(p.x, p.y)]))[0]

        if val > NODATA_THRESHOLD:
            rows.append({
                "station_ft": station_ft,
                "elev_ft": float(val),
            })

    return rows

def get_grade_smoothed(line: LineString, src, n: int = N_SAMPLES, window: int = SMOOTH_WINDOW):
    """Return (grade_range_smooth, grade_median, grade_p90, grade_max)."""
    prof = sample_profile_along_line(line, src, n=n)
    if len(prof) < 2 or line.length == 0:
        return np.nan, np.nan, np.nan, np.nan

    g = pd.DataFrame(prof).sort_values("station_ft")
    g["elev_smooth_ft"] = (
        g["elev_ft"]
        .rolling(window=window, center=True, min_periods=2)
        .median()
    ).fillna(g["elev_ft"])

    dx = g["station_ft"].diff()
    dz = g["elev_smooth_ft"].diff()
    interval_grades = (dz / dx).abs().replace([np.inf, -np.inf], np.nan).dropna()

    # Main model/app field: total smoothed elevation range divided by segment length.
    grade_range_smooth = (
        g["elev_smooth_ft"].max() - g["elev_smooth_ft"].min()
    ) / line.length

    if len(interval_grades) == 0:
        return grade_range_smooth, np.nan, np.nan, np.nan

    return (
        grade_range_smooth,
        interval_grades.median(),
        interval_grades.quantile(0.90),
        interval_grades.max(),
    )


In [ ]:
with rasterio.open(elevation_raster_path) as src:
    results = center_line_master_2722.geometry.apply(
        lambda line: get_grade_smoothed(line, src, n=N_SAMPLES, window=SMOOTH_WINDOW)
    )

center_line_master_2722["grade_range_smooth"] = results.apply(lambda x: x[0])
center_line_master_2722["grade_smooth_median"] = results.apply(lambda x: x[1])
center_line_master_2722["grade_smooth_p90"] = results.apply(lambda x: x[2])
center_line_master_2722["grade_smooth_max"] = results.apply(lambda x: x[3])

center_line_master_2722[[
    "seg_id",
    "grade_range_smooth",
    "grade_smooth_median",
    "grade_smooth_p90",
    "grade_smooth_max",
]].head()


The final analytical table carries all four grade fields, but the main map/model field is `grade_range_smooth`. In plain terms, it is the segment's smoothed elevation range divided by segment length. Values around 0.03 are noticeable hills, values around 0.05 are steep, and very high values remain a data-quality warning to inspect bridges, ramps, overpasses, and very short fragments.


In [ ]:
print("PPD Street Trees ")
tree_data = gpd.read_file("raw-data/Phila Street Trees/ppr_tree_inventory_2024.geojson")
print(tree_data.crs) # 4326
tree_data.plot(cmap="Greens", ax=None, figsize=(20,20))

In [ ]:
print("Census info: ")

block_groups_path = "raw-data/Census/phila_block_groups_2024_4269.geojson"

if not os.path.exists(block_groups_path):
    block_groups = pygris.block_groups(state="PA", county="Philadelphia", year=2024)
    block_groups.to_file(block_groups_path, driver="GeoJSON")
else:
    block_groups = gpd.read_file(block_groups_path)

print(len(block_groups))
block_groups.head(10)
print(block_groups.crs) # 4269

census_api_key = os.environ.get("CENSUS_API_KEY")
if not census_api_key:
    raise RuntimeError("Set CENSUS_API_KEY before running Census cells")

cen = census.Census(census_api_key)

tract_data = cen.acs5.state_county_tract(
    fields=("NAME",
    "B19013_001E",  # median income
    "B17001_002E",  # poverty - hmmm None...
    "B01003_001E",  # population
    "B08301_001E",  # total commuters
    "B08301_003E",  # drove alone
    "B08301_010E",  # transit
    "B08301_019E",  # walked
    "B08301_021E",  # WFH
    "B02001_002E",  # white alone
    "B25002_003E",  # vacant units
    "B25058_001E",  # median rent
    "B01002_001E"),  # median age
    state_fips='42',
    county_fips="101",
    tract="*",
    year=2024
)


tract_data_df = pd.DataFrame(tract_data)
print("Tract: \n")
print(tract_data_df.head(10))

blockgroup_data = cen.acs5.state_county_blockgroup(
    fields=("NAME",
    "B19013_001E",  # median income
    "B01003_001E",  # population
    "B08301_001E",  # total commuters
    "B08301_003E",  # drove alone
    "B08301_010E",  # transit
    "B08301_019E",  # walked
    "B08301_021E",  # WFH
    "B02001_002E",  # white alone
    "B25002_003E",  # vacant units
    "B25058_001E",  # median rent
    "B01002_001E"),  # median age
    state_fips='42',
    county_fips="101",
    blockgroup="*",
    year=2024
)

blockgroup_data_frame = pd.DataFrame(blockgroup_data)
print("Blockgroup: \n")

print(blockgroup_data_frame.head(10))

In [ ]:
print("Land use")

land_usage = gpd.read_file("raw-data/Philly Land Use/Land_Use.geojson")
print(land_usage.crs) # 4326

## Statistical Analysis and Models

The final modeling step uses the completed analytical table, one row per driveable street centerline segment. The response variable is `crash_count`, modeled as a count with a negative binomial GLM. I use `log_length` as an offset so longer segments are not treated the same as shorter ones.

The three-model structure matches the presentation:

1. **Model 1:** road design and exposure only.
2. **Model 2:** add environmental variables: tree canopy and grade.
3. **Model 3:** add interactions: canopy by cartway width, and grade by speed.


In [ ]:
model_data = gpd.read_file("transformed-data/Stash or final/philly_final_analytical_table.gpkg")

def boolish_to_int(series):
    return series.map({True: 1, False: 0, "True": 1, "False": 0}).fillna(series)

for field in ["is_divided", "has_calming"]:
    if field in model_data.columns:
        model_data[field] = boolish_to_int(model_data[field])

cols_min = [
    "crash_count",
    "cartway_width_ft",
    "lanes_final",
    "maxspeed_final",
    "is_divided",
    "has_signal",
    "has_calming",
    "calming_device_count",
    "dvrpc_aadt",
    "canopy_pct",
    "grade_range_smooth",
    "median_income",
    "length",
    "class",
]

print(model_data[cols_min].isna().sum().sort_values(ascending=False))


In [ ]:
# Lane coverage is limited, so state-road lane data falls back to centerline class.
lanes_by_class = {
    1: 4,  # Expressway
    2: 4,  # Major Arterial
    3: 2,  # Minor Arterial
    4: 2,  # Collector
    5: 2,  # Local
    9: 1,  # Low Speed Ramp
    10: 2, # High Speed Ramp
}

model_data["lanes_final"] = model_data["lanes_final"].fillna(
    model_data["class"].map(lanes_by_class)
)

# Missing divided-road values mostly indicate non-state roads in this table.
model_data["is_divided"] = model_data["is_divided"].fillna(0)

# DVRPC AADT has sparse coverage. Keep a coverage flag and median-fill the value.
model_data["has_aadt"] = model_data["dvrpc_aadt"].notna().astype(int)
model_data["dvrpc_aadt"] = model_data["dvrpc_aadt"].fillna(
    model_data["dvrpc_aadt"].median()
)

# Cartway width is mostly complete; fill remaining gaps by centerline class.
model_data["cartway_width_ft"] = model_data["cartway_width_ft"].fillna(
    model_data.groupby("class")["cartway_width_ft"].transform("median")
)

model_data["has_calming"] = (model_data["calming_device_count"] > 0).astype(int)

# The few segments without block-group context are boundary/industrial-edge cases.
model_data = model_data.dropna(subset=["median_income"])

cols_final = [
    "crash_count",
    "cartway_width_ft",
    "lanes_final",
    "maxspeed_final",
    "is_divided",
    "has_signal",
    "has_calming",
    "dvrpc_aadt",
    "has_aadt",
    "canopy_pct",
    "grade_range_smooth",
    "length",
    "class",
]

model_df = model_data[cols_final].copy()
model_df["log_aadt"] = np.log1p(model_df["dvrpc_aadt"])
model_df["log_length"] = np.log(model_df["length"])
model_df["log_grade"] = np.log1p(model_df["grade_range_smooth"])

model_df.describe()


### Why Negative Binomial?

Crash counts are sparse, non-negative count data, not continuous outcomes. A Poisson model would assume equal mean and variance, which is too restrictive for this kind of crash distribution. I used a negative binomial GLM because it allows overdispersion while keeping the model interpretable.


In [ ]:
m1 = smf.glm(
    formula=(
        "crash_count ~ cartway_width_ft + lanes_final + maxspeed_final "
        "+ is_divided + has_signal + has_calming "
        "+ log_aadt + has_aadt"
    ),
    data=model_df,
    family=sm.families.NegativeBinomial(),
    offset=model_df["log_length"],
).fit()

m2 = smf.glm(
    formula=(
        "crash_count ~ cartway_width_ft + lanes_final + maxspeed_final "
        "+ is_divided + has_signal + has_calming "
        "+ log_aadt + has_aadt "
        "+ canopy_pct + log_grade"
    ),
    data=model_df,
    family=sm.families.NegativeBinomial(),
    offset=model_df["log_length"],
).fit()

m3 = smf.glm(
    formula=(
        "crash_count ~ cartway_width_ft + lanes_final + maxspeed_final "
        "+ is_divided + has_signal + has_calming "
        "+ log_aadt + has_aadt "
        "+ canopy_pct + log_grade "
        "+ canopy_pct:cartway_width_ft "
        "+ log_grade:maxspeed_final"
    ),
    data=model_df,
    family=sm.families.NegativeBinomial(),
    offset=model_df["log_length"],
).fit()

print("MODEL 1: Road design only")
print(m1.summary())
print("MODEL 2: Add environmental")
print(m2.summary())
print("MODEL 3: Add interactions")
print(m3.summary())


In [ ]:
def lr_test(m_small, m_big, label):
    lr_stat = 2 * (m_big.llf - m_small.llf)
    df_diff = m_big.df_model - m_small.df_model
    p = 1 - chi2.cdf(lr_stat, df_diff)
    return {"Comparison": label, "LR": lr_stat, "df": df_diff, "p": p}

comparison = pd.DataFrame({
    "Model": ["M1: Road design", "M2: + Environment", "M3: + Interactions"],
    "AIC": [m1.aic, m2.aic, m3.aic],
    "Log-Likelihood": [m1.llf, m2.llf, m3.llf],
    "Pseudo-R2": [
        1 - m1.deviance / m1.null_deviance,
        1 - m2.deviance / m2.null_deviance,
        1 - m3.deviance / m3.null_deviance,
    ],
})

lr_results = pd.DataFrame([
    lr_test(m1, m2, "M1 vs M2: environment adds explanatory power"),
    lr_test(m2, m3, "M2 vs M3: interactions add explanatory power"),
])

display(comparison)
display(lr_results)


### Model Findings

The presentation findings came from these three negative binomial models. In the saved run, Model 1 had a Cox-Snell pseudo-R2 of about **0.3193**. Wider cartways, more inferred lanes, higher speed limits, traffic signals, traffic-calming presence, and AADT were positively associated with crash counts. The traffic-calming result is likely reactive: calming devices are often installed where crash risk is already known.

Model 2 improved modestly, to about **0.3218**, after adding canopy and grade. Both environmental variables were statistically significant and negatively associated with crashes: `canopy_pct = -0.7914` and `log_grade = -2.7630` in the saved run.

Model 3 improved again, to about **0.3233**, after adding interactions. The `canopy_pct:cartway_width_ft` interaction was positive, meaning canopy's apparent protective relationship is strongest on narrower streets and weakens as roads get wider. The `log_grade:maxspeed_final` interaction was also positive, meaning grade looks less protective, and potentially more hazardous, as speed context increases.

Likelihood-ratio tests in the saved run showed both added blocks improved fit: environment over road-design only, and interactions over the additive environment model. These remain correlational findings, not causal estimates.


## Where the App Continues

This notebook stops at the analysis and data-preparation stage. The final app work happens in `app/`:

- `app/src/db.ts` initializes DuckDB-WASM and registers the parquet files.
- `app/src/mapQueries.ts` contains the browser-side SQL used by the map, peer comparisons, and story examples.
- `app/src/mapLayers.ts` and `app/src/map.ts` define the MapLibre sources, layers, legends, boundary, and interactions.

The notebook is still useful as the provenance trail: it shows how I inspected the raw public datasets, made field decisions, and checked which inputs were worth carrying forward.
The final app is hosted here: https://benpolinsky.github.io/PHLCRSH/.
